# run_simulation_v2.ipynb — replicated ablation experiment

Runs the 4-variant ablation matrix with **5 replicates per variant** (20 runs per model family), in parallel.
Run this notebook **once per model family** (set `MODEL_FAMILY` in Section 2).

| Variant | Models (`claude` family) | Models (`openai` family) | Ablation |
|---|---|---|---|
| `Baseline` | claude-sonnet-4-6 | openai/gpt-5.4 | Full (memory + retrieval + reflection) |
| `Ablation1_No_Reflection` | claude-sonnet-4-6 | openai/gpt-5.4 | Memory + retrieval, no reflection |
| `Ablation2_No_Memory_No_Reflection` | claude-sonnet-4-6 | openai/gpt-5.4 | Seed personality only |
| `Budget` | claude-haiku-4-5 | openai/gpt-5.4-mini | Full system, budget model |

The judge (used later by `run_evaluation_v2.ipynb`) is always the **opposite** family:
GPT-5.4 judges Claude simulations, Claude Opus 4.6 judges OpenAI simulations.

**Sections:**
1. Setup
2. Experiment configuration (`MODEL_FAMILY`, replicates, parallelism)
3. First run once — verify the pipeline (counts as Baseline replicate 1, so it isn't wasted)
4. Full matrix in parallel — skips runs that already completed, so it's safe to re-run after failures

Each run writes `outputs/runs/<run_id>.jsonl` with `run_label = <family>_<variant>_rep<N>`.


In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic openai \
  sentence-transformers \
  'numpy>=1.26' \
  'pandas>=2.2' \
  openpyxl \
  pyyaml

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION/berkeley-homes-wildfire-agent-simulation'
except ImportError:
    PROJECT_PATH = os.path.abspath('..')   # running locally: one level up from notebooks/

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

sys.path.insert(0, PROJECT_PATH)

from src.llm.client import (
    Config, UsageTracker, use_tracker,
    init_clients, init_openrouter_client, embed,
)
from src.engine.simulation import Simulation, SimulationConfig
from src.agents.retrieval import RetrievalConfig
from src.agents.reflection import ReflectionConfig

embed('warmup')   # preload the embedding model BEFORE parallel execution (avoids a lazy-init race)
print('Imports OK')

In [ ]:
# Both clients are required: OpenRouter serves the GPT judge and the OpenAI simulation models.
client_anthropic  = init_clients()
client_openrouter = init_openrouter_client()

---
## 2. Experiment configuration

In [ ]:
# ── Which model family to simulate ────────────────────────────────────────────
# Run the notebook once with 'claude', then change to 'openai' and run again.
MODEL_FAMILY = 'claude'          # 'claude' | 'openai'

N_REPLICATES = 5                 # simulation runs per variant
MAX_PARALLEL = 4                 # concurrent simulation runs (each run is itself sequential)
FORCE_RERUN  = False             # True -> redo runs even if a completed JSONL already exists

# Judge is always the opposite family to the simulation models (self-preference bias control).
FAMILY_MODELS = {
    'claude': {'base':   'claude-sonnet-4-6',
               'budget': 'claude-haiku-4-5-20251001',
               'judge':  'openai/gpt-5.4'},
    'openai': {'base':   'openai/gpt-5.4',
               'budget': 'openai/gpt-5.4-mini',
               'judge':  'claude-opus-4-6'},
}

SCENARIO_PATH = 'config/scenarios/baseline.yaml'
AGENT_PATHS = [
    'config/agents/selected/jennifer.yaml',
    'config/agents/selected/beth.yaml',
    'config/agents/selected/edward.yaml',
    'config/agents/selected/lola.yaml',
    'config/agents/selected/synthetic_non_compliant.yaml',
]

# Locked from agent validation (agent_validation/agent_validation_beth.ipynb)
RETRIEVAL_CFG  = RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=2.0)
REFLECTION_CFG = ReflectionConfig(threshold=50.0, num_questions=3)

# GPT-5.x are reasoning models: hidden reasoning tokens count against max_tokens on
# OpenRouter, so the Claude-tuned caps (e.g. IMPORTANCE_MAX_TOKENS=5) would come back
# empty. Give the OpenAI family generous caps — billing is by actual usage, not the cap.
FAMILY_LLM_LIMITS = {
    'claude': {},
    'openai': {
        'DECISION_MAX_TOKENS':            4096,
        'REFLECTION_MAX_TOKENS':          4096,
        'REFLECTION_QUESTION_MAX_TOKENS': 4096,
        'IMPORTANCE_MAX_TOKENS':          2048,
    },
}

VARIANTS = [
    {'name': 'Baseline',                          'tier': 'base',   'use_memory': True,  'use_reflection': True},
    {'name': 'Ablation1_No_Reflection',           'tier': 'base',   'use_memory': True,  'use_reflection': False},
    {'name': 'Ablation2_No_Memory_No_Reflection', 'tier': 'base',   'use_memory': False, 'use_reflection': False},
    {'name': 'Budget',                            'tier': 'budget', 'use_memory': True,  'use_reflection': True},
]

_models = FAMILY_MODELS[MODEL_FAMILY]
print(f"Family: {MODEL_FAMILY}")
print(f"  base model:   {_models['base']}")
print(f"  budget model: {_models['budget']}")
print(f"  judge model:  {_models['judge']} (used in run_evaluation_v2.ipynb)")
print(f"{len(VARIANTS)} variants x {N_REPLICATES} replicates = {len(VARIANTS) * N_REPLICATES} runs")

In [ ]:
def make_label(variant_name: str, rep: int) -> str:
    return f"{MODEL_FAMILY}_{variant_name}_rep{rep}"


def build_sim_config(variant: dict, rep: int) -> SimulationConfig:
    models = FAMILY_MODELS[MODEL_FAMILY]
    decision_model = models[variant['tier']]
    return SimulationConfig(
        run_label        = make_label(variant['name'], rep),
        scenario_path    = SCENARIO_PATH,
        agent_yaml_paths = AGENT_PATHS,
        llm_config       = Config(
            DECISION_MODEL   = decision_model,
            REFLECTION_MODEL = decision_model,
            JUDGE_MODEL      = models['judge'],
            CONCISE_OUTPUT   = True,
            **FAMILY_LLM_LIMITS[MODEL_FAMILY],
        ),
        retrieval_config  = RETRIEVAL_CFG,
        reflection_config = REFLECTION_CFG,
        use_memory        = variant['use_memory'],
        use_reflection    = variant['use_reflection'],
    )


def completed_labels() -> set:
    """run_labels in outputs/runs that finished (JSONL contains a run_summary entry)."""
    labels = set()
    for path in Path('outputs/runs').glob('*.jsonl'):
        label, finished = None, False
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    entry = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if entry.get('entry_type') == 'run_config':
                    label = entry.get('run_label')
                elif entry.get('entry_type') == 'run_summary':
                    finished = True
        if label and finished:
            labels.add(label)
    return labels


def run_one(variant: dict, rep: int, verbose: bool = False) -> dict:
    """Run one full simulation with its own usage tracker (parallel-safe)."""
    sim_config = build_sim_config(variant, rep)
    tracker = UsageTracker()
    start = time.time()
    with use_tracker(tracker):
        sim = Simulation(
            sim_config        = sim_config,
            client_anthropic  = client_anthropic,
            client_openrouter = client_openrouter,
        )
        sim.run(verbose=verbose)
    latency = time.time() - start
    cost_info = tracker.to_dict(
        agent_model = sim_config.llm_config.DECISION_MODEL,
        judge_model = sim_config.llm_config.JUDGE_MODEL,
    )
    sim.logger.log_run_summary(latency_seconds=latency, cost_info=cost_info)
    sim.logger.close()
    return {
        'run_label':      sim_config.run_label,
        'run_id':         sim_config.run_id,
        'variant':        variant['name'],
        'rep':            rep,
        'latency_s':      round(latency, 1),
        'agent_cost_usd': cost_info['agent_cost_usd'],
        'jsonl':          f"outputs/runs/{sim_config.run_id}.jsonl",
    }

print('Helpers ready')

---
## 3. First run once — verify the pipeline

Runs Baseline replicate 1 on its own, with verbose output, so you can sanity-check decisions/reasoning
before committing to the full matrix. It writes a normal JSONL, so it **counts as replicate 1** — the
full-matrix cell below will see it as complete and not re-run it.

In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    _label = make_label(VARIANTS[0]['name'], 1)
    if not FORCE_RERUN and _label in completed_labels():
        print(f'{_label} already completed — skipping (set FORCE_RERUN=True to redo).')
    else:
        smoke_summary = run_one(VARIANTS[0], rep=1, verbose=True)
        print('\nSmoke run complete:')
        for k, v in smoke_summary.items():
            print(f'  {k}: {v}')

---
## 4. Full matrix — all variants × replicates, in parallel

- Runs up to `MAX_PARALLEL` simulations concurrently (each run has its own cost tracker, so per-run costs stay accurate).
- **Resumable:** completed runs are detected from `outputs/runs/` and skipped — if anything fails, just re-run this cell.

In [ ]:
jobs = [(variant, rep) for variant in VARIANTS for rep in range(1, N_REPLICATES + 1)]
done = set() if FORCE_RERUN else completed_labels()
pending = [(v, r) for v, r in jobs if make_label(v['name'], r) not in done]

print(f'{len(jobs)} total runs | {len(jobs) - len(pending)} already complete | {len(pending)} to run\n')

run_summaries, failures = [], []
if pending:
    matrix_start = time.time()
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
        futures = {pool.submit(run_one, v, r): make_label(v['name'], r) for v, r in pending}
        for future in as_completed(futures):
            label = futures[future]
            try:
                summary = future.result()
                run_summaries.append(summary)
                print(f"DONE   {label} | {summary['latency_s']}s | agent=${summary['agent_cost_usd']:.4f}")
            except Exception as exc:
                failures.append({'run_label': label, 'error': repr(exc)})
                print(f'FAILED {label}: {exc!r}')
    print(f'\nMatrix finished in {time.time() - matrix_start:.0f}s wall-clock')

if run_summaries:
    df_runs = pd.DataFrame(run_summaries).sort_values(['variant', 'rep'])
    print('\nThis session:')
    print(df_runs[['run_label', 'latency_s', 'agent_cost_usd', 'jsonl']].to_string(index=False))
    print(f"\nSession agent cost: ${df_runs['agent_cost_usd'].sum():.2f}")

if failures:
    print('\nFAILED RUNS — re-run this cell to retry just these (completed runs are skipped):')
    print(pd.DataFrame(failures).to_string(index=False))

In [ ]:
# Completeness check across everything on disk (including previous sessions)
done_now = completed_labels()
rows = []
for variant in VARIANTS:
    for rep in range(1, N_REPLICATES + 1):
        rows.append({'variant': variant['name'], 'rep': rep,
                     'complete': make_label(variant['name'], rep) in done_now})
status = pd.DataFrame(rows).pivot(index='variant', columns='rep', values='complete')
print(f'Run matrix completeness ({MODEL_FAMILY} family):')
print(status.to_string())
if status.all().all():
    print('\nAll runs complete. Next: run_evaluation_v2.ipynb with the same MODEL_FAMILY.')